<!-- [intro-a] -->

# ParallelProse — v2, Mechanics (Comparing Two Books)

Pairs with `v2-01`: that notebook named six new pieces at arm's length, no code. This one gets
concrete about each one — real v1 code as the anchor (`agent.py`, `mcp_tools.py`, exact line
numbers), and the shapes being proposed for what replaces or extends it.

One real difference from `v1-02`'s style: nothing below is wired into `agent.py` yet. `v1-02`
verified a function that already existed by running it. There is no v2 function to run yet — this
is the design sketch that precedes writing one. Where a cell *can* run standalone without an API
key or a live agent (a type shape, a mock state, a routing function against fake data), it does,
so the shapes here are at least checked for basic soundness. Where a cell shows the eventual real
call (an LLM invocation, a live retrieval), it's commented out and marked as a sketch, not
executed.

<!-- [B-a] -->

## B — The nested state, concretely

Real anchor: `ReflectionState` in `agent.py:19-27` — one flat `TypedDict`, one corpus's worth of
fields.

In [ ]:
# [B.1a]
# Today, agent.py:19-27 — flat, one corpus:
#
# class ReflectionState(TypedDict):
#     query: str
#     narrowed_query: str | None
#     answer: str
#     needs_revision: bool
#     feedback: str
#     chunks: list[str]
#     chunks_id: int
#     iteration: int

from typing import Literal, TypedDict


class CorpusState(TypedDict):
    chunks: list[str]
    chunks_id: int
    narrowed_query: str | None
    status: Literal["ok", "retrieval_miss", "vocabulary_mismatch", "silent"]  # section E's diagnosis lands here


class ReflectionStateV2(TypedDict):
    query: str
    corpora: dict[str, CorpusState]  # keyed "A"/"B" — not hardcoded to exactly 2
    answer: dict  # structured, see section F
    feedback: str
    iteration: int


mock_state: ReflectionStateV2 = {
    "query": "what does each book say about fortune?",
    "corpora": {
        "A": {"chunks": ["..."], "chunks_id": 1, "narrowed_query": None, "status": "ok"},
        "B": {"chunks": [], "chunks_id": 0, "narrowed_query": None, "status": "retrieval_miss"},
    },
    "answer": {},
    "feedback": "",
    "iteration": 1,
}
print(mock_state["corpora"]["B"]["status"])

<!-- [B.2a] -->

Every node now reaches through one extra layer — `state["corpora"]["A"]["chunks"]` instead of
`state["chunks"]`. A small accessor absorbs that instead of every node reaching into the dict by
hand:

In [ ]:
# [B.3a]
def corpus(state: ReflectionStateV2, corpus_id: str) -> CorpusState:
    return state["corpora"][corpus_id]


print(corpus(mock_state, "A")["chunks"])

<!-- [C-a] -->

## C — Query-splitting + terminology-bridging, concretely

Real anchor: `retriever()` in `agent.py:41-52` — takes exactly one string
(`state["narrowed_query"]` or `state["query"]`), makes one MCP call with it.

In [ ]:
# [C.1a]
from pydantic import BaseModel


class SplitQueries(BaseModel):
    """One rephrased query per corpus — output of the new node that runs before retrieval."""

    per_corpus: dict[str, str]  # e.g. {"A": "...", "B": "..."}


def bridge_terminology(concept: str, corpus_id: str) -> str:
    """Looks up corpus_id's period-appropriate term for `concept`.

    e.g. bridge_terminology("virtue", "aristotle") -> "arete"
    Sketch only — not implemented. Would sit inside the query-splitting step, not replace it.
    """
    ...


# toy check that the shape holds together — no LLM call, just the container
split = SplitQueries(per_corpus={"A": "what does A say about virtue?", "B": "what does B say about arete?"})
print(split.per_corpus["B"])

<!-- [C.2a] -->

This step's output lands in each corpus's slot from section B (`corpora[X]["narrowed_query"]` or
a new field next to it) before `retriever` — section D — ever runs. `retriever` itself doesn't
change what it does with a query, only which query it gets handed, and it gets handed one per
corpus now instead of one for the whole state.

<!-- [D-a] -->

## D — The tool-selecting retriever, concretely

Real anchor: this mechanism already exists and already runs — `mcp_tools.py:125`, the demo
agent. It just isn't plugged into the reflection graph.

In [ ]:
# [D.1a]
# Already real, mcp_tools.py:125 — the exact call this borrows from:
#
# agent = create_agent(model=ChatOpenAI(model="gpt-4o-mini"), tools=bridged_tools)
# result = await agent.ainvoke({"messages": [HumanMessage(content=query)]})

# Verified real (no API key needed to construct this — it only bounds tool calls, doesn't call one):
from langchain.agents.middleware import ToolCallLimitMiddleware

retrieval_call_limit = ToolCallLimitMiddleware(thread_limit=4, exit_behavior="end")
print(retrieval_call_limit)

# Sketch — retriever becomes this, run once per corpus, each scoped to that corpus's own
# collection's tools (not executed here, no live model/tools in this notebook):
#
# def make_retrieval_agent(llm, tools_for_corpus):
#     return create_agent(model=llm, tools=tools_for_corpus, middleware=[retrieval_call_limit])

<!-- [D.2a] -->

The picking-among-tools mechanism isn't new engineering — `create_agent` already does it, proven
in `mcp_tools.py`'s demo agent. What's new is running one of these per corpus (each with its own
4-tool set, scoped to that corpus's collection) instead of running it once, standalone, to prove
the bridge works.

<!-- [E-a] -->

## E — The diagnostic Critique, concretely

Real anchor: `Critique` in `agent.py:30-38` — one `needs_revision: bool`, one `feedback`, one
optional `narrowed_query`. `reflect()` (`agent.py:81-92`) fills it from one query, one answer.

In [ ]:
# [E.1a]
# Today, agent.py:30-38 — one verdict, whole-state:
#
# class Critique(BaseModel):
#     needs_revision: bool
#     feedback: str
#     narrowed_query: str | None = None

from typing import Literal

from pydantic import BaseModel, Field


class CorpusCritique(BaseModel):
    status: Literal["ok", "retrieval_miss", "vocabulary_mismatch", "silent"] = Field(
        description="retrieval_miss/vocabulary_mismatch are worth retrying; silent means the "
        "book genuinely doesn't address this — no retry helps"
    )
    feedback: str
    narrowed_query: str | None = None


class CritiqueV2(BaseModel):
    per_corpus: dict[str, CorpusCritique]


# toy — no LLM call, just checking the shape holds two independent verdicts at once
critique = CritiqueV2(
    per_corpus={
        "A": CorpusCritique(status="ok", feedback="covers it directly"),
        "B": CorpusCritique(status="silent", feedback="book never addresses this topic"),
    }
)
print(critique.per_corpus["B"].status)

<!-- [F-a] -->

## F — The structured composer output, concretely

Real anchor: `composer()` in `agent.py:55-78` — three branches of hand-built f-string prompts,
one narrative `answer` string out.

In [ ]:
# [F.1a]
# Today, agent.py:64-75 — three f-string branches, all producing one prose "answer":
#
# if state["narrowed_query"] and state["needs_revision"]:
#     prompt = f"Given this context {context}, the query {state['narrowed_query']}, ..."
# elif state["needs_revision"]:
#     prompt = f"Given this context {context}, the query {state['query']} ..."
# else:
#     prompt = f"Given this context {context}, write a short (3-4 sentence) factual note on: ..."

from pydantic import BaseModel


class ComparisonDraft(BaseModel):
    agreement: str
    disagreement: str
    unique_to_a: str
    unique_to_b: str
    silent_on: list[str]  # e.g. ["A"] when a corpus doesn't address the topic at all


# toy — shows what E's "silent" verdict lets composer state outright, as a finding
draft = ComparisonDraft(
    agreement="",
    disagreement="",
    unique_to_a="",
    unique_to_b="B argues that virtue is trainable through habit.",
    silent_on=["A"],
)
print(draft.silent_on)

<!-- [F.2a] -->

These explicit slots are also what make E's per-corpus check possible to grade at all — `reflect`
can look at `unique_to_a` and `silent_on` directly instead of parsing one paragraph to guess
whether Book A's material actually made it in.

<!-- [G-a] -->

## G — Per-corpus retry routing, concretely

Real anchor: `route_after_reflection()` in `agent.py:95-101` — one flat check, one target.

In [ ]:
# [G.1a]
# Verified, not assumed: LangGraph's conditional-edge path function is allowed to return more
# than one node name — it fans out to all of them. Checked against the installed version, not
# the docs alone:
import inspect

from langgraph.graph import StateGraph

print(inspect.signature(StateGraph.add_conditional_edges))
print(inspect.getdoc(StateGraph.add_conditional_edges))

In [ ]:
# [G.2a]
# Today, agent.py:95-101 — one target, whole-state:
#
# def route_after_reflection(state: ReflectionState):
#     if not state["needs_revision"] or state["iteration"] >= MAX_ITERATIONS:
#         return END
#     if state["narrowed_query"]:
#         return "retriever"
#     return "composer"

from langgraph.graph import END

MAX_ITERATIONS = 5


def route_after_reflection_v2(state: ReflectionStateV2):
    if state["iteration"] >= MAX_ITERATIONS:
        return END
    retry_targets = [
        cid
        for cid, c in state["corpora"].items()
        if c["status"] in ("retrieval_miss", "vocabulary_mismatch")
    ]
    return retry_targets or END  # fans out to just the flagged corpora, per G.1a


# toy check, no live graph needed
print(route_after_reflection_v2(mock_state))  # mock_state's "B" is retrieval_miss

<!-- [G.2b] -->

`route_after_reflection_v2` returning `["B"]` instead of one hardcoded string is the entire
mechanism — only the flagged corpus re-enters `retriever` (section D) with the fix from section C
applied to it; corpus A's slot in the nested state (section B) is never touched.

<!-- [close-a] -->

## Recap

Six sketches, each anchored to the exact v1 code it extends or replaces:

- **B** — `ReflectionState` (flat) -> `ReflectionStateV2` (nested `corpora` dict) + an accessor.
- **C** — `retriever`'s single query -> `SplitQueries` + a `bridge_terminology` hook, per corpus.
- **D** — `retriever`'s hardcoded MCP call -> `mcp_tools.py`'s already-proven `create_agent`
  pattern, bounded by `ToolCallLimitMiddleware`, run once per corpus.
- **E** — `Critique` (one verdict) -> `CritiqueV2` (`CorpusCritique` per corpus, a status label
  instead of a bool).
- **F** — `composer`'s f-string branches -> `ComparisonDraft`'s explicit slots, silence included.
- **G** — `route_after_reflection`'s one target -> a list of flagged corpus ids, verified to fan
  out correctly against the real `add_conditional_edges` signature.

Nothing here is wired into `agent.py` yet. The next step is implementation, not another notebook
— `ROADMAP.md`'s ordering (state shape and the merge first, composer output last) is the sequence
these six sketches were deliberately built to match.